# Setup Guide — Thesis Experiment Environment

This notebook walks you through setting up both experiment environments:

1. **UTM VM** — Ubuntu 22.04, 4 vCPU, 8 GB RAM, Dagster + PostgreSQL (Experiment 1 / SQ1)
2. **Kind cluster** — Local Kubernetes via podman (rootful), Dagster via Helm (Experiment 2A / SQ2-SQ3)

> **Run cells top-to-bottom.**  Commands note `[host]` (your Mac) or `[vm]` (inside the VM).
>
> PDF compilation uses **Overleaf** — `make pdf` will print instructions; no local LaTeX needed.


## 1 — Prerequisites check

Verify required tools are installed on your Mac.

In [4]:
import subprocess

tools = {
    "podman":  "brew install podman",
    "kind":    "brew install kind",
    "kubectl": "brew install kubectl",
    "helm":    "brew install helm",
    "jq":      "brew install jq",
    "gh":      "brew install gh",
    "ansible": "pip3 install ansible",
}

all_ok = True
for tool, hint in tools.items():
    r = subprocess.run(["which", tool], capture_output=True, text=True)
    if r.returncode == 0:
        ver = subprocess.run([tool, "--version"], capture_output=True, text=True)
        v = ver.stdout.strip().splitlines()[0] if ver.returncode == 0 else ""
        print(f"  OK  {tool:<12} {r.stdout.strip()}  ({v})")
    else:
        print(f"  !!  {tool:<12} NOT FOUND  ->  {hint}")
        all_ok = False

print()
print("All prerequisites met!" if all_ok else "Install missing tools then re-run this cell.")


  OK  podman       /opt/homebrew/bin/podman  (podman version 5.7.1)
  OK  kind         /opt/homebrew/bin/kind  (kind version 0.30.0)
  OK  kubectl      /opt/homebrew/bin/kubectl  ()
  OK  helm         /opt/homebrew/bin/helm  ()
  OK  jq           /usr/bin/jq  (jq-1.7.1-apple)
  OK  gh           /opt/homebrew/bin/gh  (gh version 2.89.0 (2026-03-26))
  OK  ansible      /opt/homebrew/bin/ansible  (ansible [core 2.20.4])

All prerequisites met!


---
## 2 — UTM VM Setup (THESIS-001)

The VM is provisioned via **Ansible** (`make vm-provision`).
If your VM already exists, skip to section 2.3 to verify it.

**UTM VM specs (required):** 4 vCPU · 8 GB RAM · 20 GB disk · Ubuntu 22.04

> **Tip:** Install UTM from https://mac.getutm.app/  
> Create a VM manually with the specs above, or use the provisioning script.


### 2.1 — Create the VM (UTM, one-time)

In [5]:
# [host] After creating the VM in UTM, write its IP to vm-ip.txt:
# echo "192.168.64.X" > vm-ip.txt     # replace X with actual UTM IP

import subprocess, os

vm_ip_file = "../vm-ip.txt"
if os.path.exists(vm_ip_file):
    ip = open(vm_ip_file).read().strip()
    print(f"VM IP: {ip}")
    result = subprocess.run(
        ["ssh", "-i", os.path.expanduser("~/.ssh/thesis_vm"),
         "-o", "StrictHostKeyChecking=no", "-o", "ConnectTimeout=5",
         f"ubuntu@{ip}", "echo OK"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("SSH reachable")
    else:
        print("SSH not reachable yet:", result.stderr.strip())
else:
    print("vm-ip.txt not found. Create your UTM VM and write its IP to vm-ip.txt first.")


VM IP: 192.168.100.100
SSH not reachable yet: ssh: connect to host 192.168.100.100 port 22: Operation timed out


### 2.2 — Provision the VM via Ansible

In [6]:
# [host] Installs Python 3.12, Docker CE, PostgreSQL 16, Dagster 1.12.7 — idempotent.

import subprocess, os

vm_ip_file = "../vm-ip.txt"
if os.path.exists(vm_ip_file):
    result = subprocess.run(
        ["make", "-C", "..", "vm-provision"],
        capture_output=True, text=True
    )
    print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
else:
    print("Skipping: vm-ip.txt not found.")


----------------------------------------------------------------
Provisioning VM via Ansible
----------------------------------------------------------------
→ Using VM IP from /Users/sirajulhaqwahaj/thesis/vm-ip.txt: 192.168.100.100
→ Generating Ansible inventory...
[OK] Inventory written to /Users/sirajulhaqwahaj/thesis/ansible/inventory.ini

→ Testing SSH connectivity to 192.168.100.100...
[FAIL] Cannot SSH to ubuntu@192.168.100.100
   Ensure the VM is running and accessible.
   Expected SSH key: /Users/sirajulhaqwahaj/.ssh/thesis_vm



### 2.3 — Verify the VM

In [7]:
import subprocess, os

vm_ip_file = "../vm-ip.txt"
if not os.path.exists(vm_ip_file):
    print("vm-ip.txt not found.")
else:
    ip = open(vm_ip_file).read().strip()
    checks = [
        ("nproc",             "CPU count"),
        ("free -h | head -2", "Memory"),
        ("python3 --version", "Python"),
        ("dagster --version", "Dagster"),
        ("docker --version",  "Docker CE"),
        ("pg_isready",        "PostgreSQL"),
    ]
    for cmd, label in checks:
        r = subprocess.run(
            ["ssh", "-i", os.path.expanduser("~/.ssh/thesis_vm"),
             "-o", "StrictHostKeyChecking=no", f"ubuntu@{ip}", cmd],
            capture_output=True, text=True, timeout=10
        )
        status = r.stdout.strip() or r.stderr.strip()
        icon = "OK" if r.returncode == 0 else "!!"
        print(f"  {icon}  {label:<15} {status}")


TimeoutExpired: Command '['ssh', '-i', '/Users/sirajulhaqwahaj/.ssh/thesis_vm', '-o', 'StrictHostKeyChecking=no', 'ubuntu@192.168.100.100', 'nproc']' timed out after 10 seconds

---
## 3 — Kind Cluster Setup (THESIS-002)

The Kind cluster uses **podman in rootful mode** as the container runtime.
The cluster is named `thesis` and hosts the K8sRunLauncher Dagster deployment.


### 3.1 — Ensure podman machine is running in rootful mode

In [9]:
import subprocess

r = subprocess.run(["podman", "machine", "list"], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else r.stderr)

print()
print("The machine used by this project is 'thesis' (rootful).")
print("If it's not listed above, run:")
print("  podman machine init --rootful thesis")
print("  podman machine start thesis")


NAME                    VM TYPE     CREATED       LAST UP            CPUS        MEMORY      DISK SIZE
thesis                  applehv     22 hours ago  Currently running  4           8GiB        60GiB
podman-machine-default  applehv     3 months ago  18 hours ago       4           2GiB        100GiB


The machine used by this project is 'thesis' (rootful).
If it's not listed above, run:
  podman machine init --rootful thesis
  podman machine start thesis


### 3.2 — Create the Kind cluster

In [10]:
import subprocess

r = subprocess.run(["kind", "get", "clusters"], capture_output=True, text=True)
clusters = r.stdout.strip().split("\n") if r.returncode == 0 else []

if "thesis" in clusters:
    print("Kind cluster 'thesis' already exists.")
else:
    print("Creating Kind cluster 'thesis' ...")
    r2 = subprocess.run(
        ["bash", "../scripts/create-kind-cluster.sh"],
        capture_output=True, text=True
    )
    print(r2.stdout)
    if r2.returncode != 0:
        print("ERROR:", r2.stderr)


Creating Kind cluster 'thesis' ...
→ Using Docker socket: unix:///var/run/docker.sock

── Kind Cluster Setup --------------------------------------------
→ Creating Kind cluster 'thesis'...

ERROR: ERROR: failed to create cluster: failed to get docker info: command "docker info --format '{{json .}}'" failed with error: exit status 1

Command Output: {"ID":"","Containers":0,"ContainersRunning":0,"ContainersPaused":0,"ContainersStopped":0,"Images":0,"Driver":"","DriverStatus":null,"Plugins":{"Volume":null,"Network":null,"Authorization":null,"Log":null},"MemoryLimit":false,"SwapLimit":false,"CpuCfsPeriod":false,"CpuCfsQuota":false,"CPUShares":false,"CPUSet":false,"PidsLimit":false,"IPv4Forwarding":false,"Debug":false,"NFd":0,"OomKillDisable":false,"NGoroutines":0,"SystemTime":"","LoggingDriver":"","CgroupDriver":"","NEventsListener":0,"KernelVersion":"","OperatingSystem":"","OSVersion":"","OSType":"","Architecture":"","IndexServerAddress":"","RegistryConfig":null,"NCPU":0,"MemTotal":0,"Ge

### 3.3 — Deploy Dagster via Helm (thesis-specific chart)

In [11]:
import subprocess

print("Deploying Dagster using the thesis Helm chart ...")
r = subprocess.run(
    ["bash", "../scripts/deploy-dagster-k8s.sh"],
    capture_output=True, text=True
)
print(r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout)
if r.returncode != 0:
    print("ERROR:", r.stderr[-1000:])


Deploying Dagster using the thesis Helm chart ...

── Dagster K8s Deployment ----------------------------------------
→ Using container runtime via: system default

ERROR: ERROR: failed to list nodes: command "docker ps -a --filter label=io.x-k8s.kind.cluster=thesis --format '{{.Names}}'" failed with error: exit status 1

Command Output: Cannot connect to the Docker daemon at unix:///Users/sirajulhaqwahaj/.docker/run/docker.sock. Is the docker daemon running?



### 3.4 — Verify K8s deployment

In [12]:
import subprocess, time, json

print("Waiting for Dagster pods ...")
for attempt in range(24):
    r = subprocess.run(
        ["kubectl", "get", "pods", "-n", "dagster",
         "--no-headers", "-o",
         "custom-columns=NAME:.metadata.name,STATUS:.status.phase"],
        capture_output=True, text=True
    )
    lines = [l for l in r.stdout.strip().splitlines() if l]
    running = sum(1 for l in lines if "Running" in l)
    total   = len(lines)
    if total > 0:
        print(f"  {running}/{total} Running  (attempt {attempt+1})")
        if running == total:
            print("All pods Running!")
            break
    time.sleep(5)
else:
    print("Timeout. Current state:")
    print(r.stdout)

pf = subprocess.Popen(
    ["kubectl", "port-forward", "-n", "dagster",
     "svc/dagster-thesis-webserver", "13999:3000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)
gql = subprocess.run(
    ["curl", "-s", "--max-time", "5", "-X", "POST",
     "http://localhost:13999/graphql",
     "-H", "Content-Type: application/json",
     "-d", json.dumps({"query": "{ version }"})],
    capture_output=True, text=True
)
pf.terminate()
print()
print("GraphQL API response:", gql.stdout.strip())


Waiting for Dagster pods ...
  4/92 Running  (attempt 1)
  4/92 Running  (attempt 2)
  4/92 Running  (attempt 3)
  4/92 Running  (attempt 4)
  4/92 Running  (attempt 5)
  4/92 Running  (attempt 6)
  4/92 Running  (attempt 7)
  4/92 Running  (attempt 8)
  4/92 Running  (attempt 9)
  4/92 Running  (attempt 10)
  4/92 Running  (attempt 11)
  4/92 Running  (attempt 12)
  4/92 Running  (attempt 13)
  4/92 Running  (attempt 14)
  4/92 Running  (attempt 15)
  4/92 Running  (attempt 16)
  4/92 Running  (attempt 17)
  4/92 Running  (attempt 18)
  4/92 Running  (attempt 19)
  4/92 Running  (attempt 20)
  4/92 Running  (attempt 21)
  4/92 Running  (attempt 22)
  4/92 Running  (attempt 23)
  4/92 Running  (attempt 24)
Timeout. Current state:
dagster-run-0010b0bb-a3e2-47d4-a8f2-bf79243c7376-rkvft   Succeeded
dagster-run-047db552-2779-4e42-a521-4ea980a402ed-jfh4f   Succeeded
dagster-run-0732e6b1-4017-4c38-b08b-43beabfbea80-xb8n5   Succeeded
dagster-run-07636698-f7d5-44f2-9c85-2ad6a27fd26b-bfp9x   Su

---
## 4 — Build & Load the Workload Image (THESIS-003)

The workload image is built from `src/Containerfile` with **podman** and loaded into Kind via `kind load`.


In [ ]:
import subprocess

REGISTRY = "localhost:5001"
IMAGE    = f"{REGISTRY}/thesis-workload:latest"

print(f"Building {IMAGE} ...")
r = subprocess.run(
    ["bash", "../scripts/02_img_build_push.sh"],
    capture_output=True, text=True, cwd=".."
)
print(r.stdout[-2000:] if len(r.stdout) > 2000 else r.stdout)
if r.returncode != 0:
    print("ERROR:", r.stderr[-500:])


---
## 5 — Smoke Test: Single L1 Run (Both Environments)

Before running full experiments, verify a single job completes on each environment.


In [ ]:
import subprocess, os

repo = os.path.abspath("..")

for env, exp in [("vm", "exp1"), ("k8s", "exp2a")]:
    r = subprocess.run(
        ["bash", "scripts/run_experiment.sh", exp, env, "--dry-run", "--levels", "1"],
        capture_output=True, text=True, cwd=repo
    )
    status = "OK" if r.returncode == 0 else "FAIL"
    print(f"[{status}] dry-run {exp} {env}")
    if r.returncode != 0:
        print("   ", r.stderr.strip()[:300])


---
## Setup Complete

If all checks above pass, you are ready to run experiments.

```bash
make exp1-vm        # Experiment 1 — VM degradation (SQ1)
make exp2a-k8s      # Experiment 2A — K8s isolation (SQ2-SQ3)
make exp2b-blast    # Experiment 2B — Blast radius (SQ2)
make exp2c-spike    # Experiment 2C — Spike observation (SQ3)
make analyze        # notebooks/analysis.ipynb -> data/processed/ + results/
```

Then upload `docs/` (with `docs/figures/`) to your **Overleaf** project and compile there.
